In [1]:
import pandas as pd
posts = pd.read_csv("/content/posts.csv")
comments = pd.read_csv("/content/reddit_comments_clean.csv")



In [2]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)


RENAMING COLUMNS

In [3]:
posts = posts.rename(columns={"id": "post_id", "text": "post_text"})
comments = comments.rename(columns={"text": "comment_text"})

FILTER OUT EMPTY TEXTS

In [4]:
posts["post_text"].isna().sum(), comments["comment_text"].isna().sum()
comments = comments.dropna(subset=["comment_text"])
posts["post_text"] = posts["post_text"].fillna("[EXTERNAL_LINK_OR_NO_TEXT]")


sort everything by time

In [5]:
posts = posts.sort_values("timestamp")
comments = comments.sort_values("timestamp")


grouping commnets by post


In [6]:
grouped_comments = comments.groupby("post_id")


BUILDING NARRATIVE UNITS EXPLICITLY

In [7]:
narrative_units = []

for _, post in posts.iterrows():
    pid = post["post_id"]

    post_text = post["post_text"]
    post_time = post["timestamp"]

    if pid in grouped_comments.groups:
        comment_texts = grouped_comments.get_group(pid)["comment_text"].tolist()
    else:
        comment_texts = []

    narrative_units.append({
        "post_id": pid,
        "post_text": post_text,
        "post_timestamp": post_time,
        "comments": comment_texts,
        "num_comments": len(comment_texts)
    })


CONVERTING NARRATIVE UNITS TO DATAFRAME


In [8]:
narratives_df = pd.DataFrame(narrative_units)
narratives_df.head()



post_id  \
0  12vyubs   
1  132oa6r   
2  134393k   
3  1345vai   
4  1347y8a   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    post_text  \
0                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            

importing transformer for sentence embedding


In [9]:
!pip install sentence-transformers nltk


note: here nltk is a natural language toolkit , will used it for sentence splitting.

punkt is a sentence tokenizer(i.e tool that splits text into meaningful units)

In [10]:
from sentence_transformers import SentenceTransformer
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

LOAD AN EMBEEDDIG MODEL

In [11]:
model = SentenceTransformer("all-MiniLM-L6-v2")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


EXTRACTING SENTENCES FROM NARRATIVE UNITS

sentence split function


In [12]:
from nltk.tokenize import sent_tokenize

def extract_sentences(text):
    if not isinstance(text, str):
        return []
    return sent_tokenize(text)


build a sentence level dataset

In [13]:
sentence_records = []

# Create lookup for post timestamps
post_time_lookup = posts.set_index("post_id")["timestamp"].to_dict()

for _, row in narratives_df.iterrows():
    post_id = row["post_id"]

    # --- Post sentences ---
    post_sentences = extract_sentences(row["post_text"])
    post_timestamp = post_time_lookup.get(post_id)

    for s in post_sentences:
        sentence_records.append({
            "post_id": post_id,
            "source": "post",
            "sentence": s,
            "timestamp": post_timestamp
        })

    # --- Comment sentences ---
    post_comments = comments[comments["post_id"] == post_id]

    for _, c in post_comments.iterrows():
        comment_sentences = extract_sentences(c["comment_text"])
        comment_timestamp = c["timestamp"]

        for s in comment_sentences:
            sentence_records.append({
                "post_id": post_id,
                "source": "comment",
                "sentence": s,
                "timestamp": comment_timestamp
            })


In [14]:
sentences_df = pd.DataFrame(sentence_records)
sentences_df.head()


,post_id,source,sentence,timestamp
0,12vyubs,post,[EXTERNAL_LINK_OR_NO_TEXT],2023-04-23 07:45:12
1,12vyubs,comment,Holy land!,2023-04-23 07:54:02
2,12vyubs,comment,Checkmate,2023-04-23 08:37:45
3,12vyubs,comment,Ke2,2023-04-23 09:33:57
4,12vyubs,comment,Wet simple… nuke it all.,2023-04-23 11:16:41


In [15]:
# SAFE RESET — clears only downstream variables, NOT sentences_df

for var in [
    "X_reduced",
    "clustered_sentences_df",
    "centroids_df",
    "narrative_map",
    "narrative_stats",
    "qualified_narratives"
]:
    if var in globals():
        del globals()[var]

print("Reset done. Raw sentences preserved.")



Reset done. Raw sentences preserved.


(issue fixing : sentence level duplication)

topic+Sentence Quality Filter

In [16]:
import re

KEYWORDS = [
    "israel", "palestine", "gaza", "hamas", "idf",
    "ceasefire", "hostage", "prisoner", "genocide",
    "occupation", "bomb", "strike", "military",
    "zionist", "settlement", "netanyahu", "biden",
    "palestinian", "israeli"
]

def is_valid_sentence(s):
    s = s.strip().lower()

    if len(s) < 40:
        return False
    if len(s.split()) < 7:
        return False

    if not any(k in s for k in KEYWORDS):
        return False

    if re.search(r"\b(lol|lmao|wtf|stan|damn)\b", s):
        return False

    return True

# Deduplicate + filter
sentences_df = sentences_df.drop_duplicates(
    subset=["sentence", "timestamp", "post_id", "source"]
)

sentences_df = sentences_df[
    sentences_df["sentence"].apply(is_valid_sentence)
].reset_index(drop=True)

print("Clean sentences:", len(sentences_df))
sentences_df.sample(10)


Clean sentences: 19995


,post_id,source,sentence,timestamp
3128,178h1mu,comment,"\n\n\nIsrael has always gone after anyone wearing a vest that says ""Press"" or ""Medic.""",2023-10-15 19:48:26
1287,173kz9y,comment,What does Hamas think it stands to gain from this?,2023-10-09 14:08:54
11141,18q310b,comment,Many Christians taking refuge in the churches were killed by Israeli airstrikes.,2023-12-24 23:03:51
2453,176p7pu,comment,"I’m pro Palestine, but Hamas are terrible, and really always have been.",2023-10-13 12:32:43
13537,1aq4k55,comment,You mean people who don't want to leave Israel?,2024-02-13 23:11:54
3743,17c04ei,comment,"Personally, I appreciate him calling what is happening genocide openly.",2023-10-20 04:05:43
17068,1bohm5p,comment,I believe Israel had a right to defend itself.,2024-03-26 22:46:16
7690,17uaptv,comment,Hamas does not care one bit about the people it represents and claims to fight for.,2023-11-13 14:57:08
13328,1am4yq5,comment,"I believe it's kinda surprising the actual Israel subreddit hasn't been brigaded yet, just a bunch of unrelated subreddits\n\n(I'm not calling for it to happen, though, just saying it's surprising)",2024-02-08 21:47:21
9725,18ctz9y,comment,"The Zionist movement was created in the age of colonialism when many European countries colonized the world and let me tell you, what Jews did to the arabs in this regeon is nothing compared to some other places",2023-12-07 16:30:34


sanity check (sentence filter)
new issue(1.1:it is removing garbage but not forming semantic clusters

In [17]:
sentences_df.sample(20)["sentence"]


,sentence
12221,Why does Palestine and that region have to pay the price for your crimes?
18510,It makes one wonder given this logic could there be a step to far for Israel?
16883,"For example the 1979 Palestinan Revolutionary code makes it explicitly illegal to normalize relations with ""the Zionist entity"" and Hamas has used that law to persecute Palestinan peace activists like Rami Amen."
16035,"Oct 7th, while absolutely indefensible, has served as the catalyst for a lot of international support and pressure being applied to Israel."
3423,Care to share why Syrian and Palestinian people were trying to kill you??
13570,"Israeli fighter jets began “an [extensive wave of attacks in Lebanese territory](https://twitter.com/idfspokesperson/status/1757733624047157511?s=46&t=RyjxkEgQcJLVY_WhzINJ3g),” Israeli Rear Adm. Daniel Hagari announced as the strikes were underway, saying details were to follow."
19830,These are psychopaths and I wonder what excuse Kenyan zionists will come up with to justify such behaviour.
17214,"[https://medium.com/@chrisjeffrieshomelessromantic/german-bank-freezes-the-account-of-the-jewish-anti-zionist-group-jewish-voice-for-a-just-peace-in-90b87c892a6a](https://medium.com/@chrisjeffrieshomelessromantic/german-bank-freezes-the-account-of-the-jewish-anti-zionist-group-jewish-voice-for-a-just-peace-in-90b87c892a6a)\n\n\n\nEchoes of History: Jewish Property and Bank Accounts in Germany\n\nIn a concerning turn of events, echoes of the past reverberate as a German bank freezes the account of the Jewish anti-Zionist group “Jewish Voice for a Just Peace in the Middle East.” The situation draws unsettling parallels to the historical persecution and confiscation of Jewish property and assets during the Nazi regime in Germany."
8868,"It wasn't that Palestine had better PR, it was that Israel just got caught lying so often that now nobody believes them even if they say something that is true."
17894,This is something you frequently hear from zionists and it's 100% false.


GENERATING EMBEDDINGS


In [18]:
embeddings = model.encode(
    sentences_df["sentence"].tolist(),
    show_progress_bar=True
)
sentences_df["embedding"] = list(embeddings)


Batches:   0%|          | 0/625 [00:00<?, ?it/s]

PHASE 4: NARRATIVE CLUSTERING & EMERGENCE





INSTALLING CLUSTERING ALGO
1. HDBSCAN
2. UMAP



In [19]:
!pip install hdbscan umap-learn


preparing embeddings for clusterings

In [20]:

import numpy as np
import hdbscan


prepare time + embeddings

In [21]:
# Ensure timestamp is datetime
sentences_df["timestamp"] = pd.to_datetime(sentences_df["timestamp"])

# Convert embeddings column to NumPy array (once)
X_all = np.vstack(sentences_df["embedding"].values)


dimensionality reduction ONCE

In [22]:
import umap

reducer = umap.UMAP(
    n_neighbors=8,          # smaller = more local structure
    n_components=50,
    min_dist=0.1,          # encourages separation
    metric="cosine",
    random_state=42
)


X_reduced = reducer.fit_transform(X_all)


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


attach reduced embeddings back


In [23]:
sentences_df["embedding_reduced"] = list(X_reduced)


72-hour sliding windows

In [24]:
from datetime import timedelta


sentences_df["timestamp"] = pd.to_datetime(sentences_df["timestamp"])

WINDOW_SIZE = timedelta(hours=72)
STEP_SIZE   = timedelta(hours=24)

start_time = sentences_df["timestamp"].min()
end_time   = sentences_df["timestamp"].max()

window_results = []
current_start = start_time

while current_start < end_time:
    current_end = current_start + WINDOW_SIZE

    window_df = sentences_df[
        (sentences_df["timestamp"] >= current_start) &
        (sentences_df["timestamp"] < current_end)
    ]

    if len(window_df) < 80:
        current_start += STEP_SIZE
        continue

    X_window = np.vstack(window_df["embedding_reduced"].values)

    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=8,
        min_samples=4,
        metric="euclidean",
        cluster_selection_method="eom"
    )

    labels = clusterer.fit_predict(X_window)

    window_df = window_df.copy()
    window_df["cluster"] = labels
    window_df["window_start"] = current_start

    # Remove noise
    window_df = window_df[window_df["cluster"] != -1]

    # Remove tiny clusters
    cluster_sizes = window_df["cluster"].value_counts()
    valid_clusters = cluster_sizes[cluster_sizes >= 8].index
    window_df = window_df[window_df["cluster"].isin(valid_clusters)]

    window_results.append(window_df)

    current_start += STEP_SIZE

clustered_sentences_df = pd.concat(window_results, ignore_index=True)

print("Total clustered sentences:", len(clustered_sentences_df))
print("Clusters per window:")
print(clustered_sentences_df.groupby("window_start")["cluster"].nunique().describe())


Total clustered sentences: 49268
Clusters per window:
count    194.000000
mean       5.664948
std        4.293157
min        2.000000
25%        2.000000
50%        4.000000
75%        8.000000
max       21.000000
Name: cluster, dtype: float64


reassemble clustered data

In [25]:
clustered_sentences_df = pd.concat(window_results, ignore_index=True)
clustered_sentences_df.head()


,post_id,source,sentence,timestamp,embedding,embedding_reduced,cluster,window_start
0,171wuh1,comment,"While Israeli conflicts with Palestinians are cause of controversy and no one is in the right, firing randomly into cities is just as bad as Russia firing cruise missiles into markets.",2023-10-07 05:15:35,"[0.1161498, 0.03151624, 0.027027603, -0.019991305, 0.020424766, 0.006404132, 0.029639963, -0.08934889, -0.03262891, 0.08873123, -0.014401015, 0.09797928, 0.04422159, 0.095378526, 0.06460649, -0.04597263, -0.034049813, -0.09819621, -0.039667614, 0.0017472205, -0.038618304, -0.026207648, 0.075134166, 0.039845977, -0.03838032, -0.030831017, 0.08905261, 0.02084684, -0.05232652, -0.053950496, 0.04231843, 0.01896746, -0.08960091, -0.057332437, 0.05533276, 0.03554423, -0.025681034, 0.050452497, -0.010265099, -0.038598377, 0.08926009, -0.012565247, 0.038672008, -0.016519701, -0.07770175, 0.026764102, -0.008833107, -0.04149561, -0.002085438, 0.0058957175, -0.051140245, 0.014949018, 0.030272158, -0.08486058, -0.004705695, -0.107464954, -0.034828134, 0.042250752, 0.051940996, -0.019271225, 0.013227606, -0.014159006, -0.07226616, -0.06629646, 0.048686873, -0.0013920786, 0.08593309, 0.07062228, -0.06483696, 0.013421553, 0.069424145, 0.06717224, -0.0125215715, 0.032175235, 0.00621064, -0.049239956, -0.020275658, 0.012196681, 0.019094054, 0.020629652, 0.074893564, -0.08874314, -0.0454074, -0.006125222, 0.09500415, 0.009285948, 0.035864588, -0.04780571, 0.10319657, 0.0035287128, -0.04867004, 0.0034987635, 0.17515646, -0.026804186, 0.06938102, 0.035644352, 0.035559718, 0.011732247, -0.0019840868, 0.009080176, ...]","[9.98577, 3.6460776, 6.0069256, 4.523595, 4.231772, 5.166909, 5.985437, 3.7605987, 5.502318, 5.172173, 5.2822666, 5.6917458, 5.3659277, 5.097566, 5.261136, 3.5542743, 2.1715593, 5.190726, 5.0367465, 3.1253867, 4.7528014, 4.7691402, 5.153231, 3.8791277, 5.2970552, 5.142651, 2.735939, 4.4951363, 2.931704, 7.2120886, 7.490108, 1.9369056, 5.0834146, 5.2483206, 3.4479609, 4.7608066, 4.3169885, 4.824349, 5.0628076, 5.082014, 5.6417785, 2.7789674, 5.676494, 5.7943044, 1.9246079, 4.1409874, 4.856318, 4.273654, 4.709491, 4.662867]",3,2023-10-04 14:16:02
1,171wuh1,comment,Hamas ~~militants~~ TERRORISTS are just driving around israel and killing everyone they see... alot of civilian casualties.,2023-10-07 06:18:29,"[0.10556259, 0.021519661, 0.035162274, -0.03638333, 0.08534444, 0.014571484, 0.065819986, -0.05803084, 0.013550521, 0.14783677, 0.04983684, -0.0042823893, 0.0816249, 0.019436382, 0.016894778, -0.031245999, -0.015107429, -0.059041444, -0.053727694, 0.013569477, -0.09563659, 0.03171039, 0.06590181, 0.0782105, 0.016069502, 0.025302835, 0.019127745, -0.007999122, 0.00010069897, -0.045608323, 0.01291985, 0.019172193, -0.09247113, 0.0039612344, 0.038346823, 0.09062769, 0.057819217, -0.038896028, -0.041968066, -0.03743754, 0.11910729, -0.0062360833, 0.030816779, -0.014611499, 0.03481742, 0.04615904, -0.033411704, -0.06735043, 0.038400676, 0.024442334, 0.031516153, -0.057457883, 0.012043201, -0.023555605, -0.00855177, -0.1539609, -0.03424632, -0.015896546, 0.013022728, -0.055631634, -0.056319073, 0.030341337, 0.06540464, -0.022432888, -0.039648112, -0.013088352, 0.07837231, 0.03241278, -0.029811092, 0.0655659, 0.042590003, 0.042863905, 0.010476511, 0.05164701, -0.02429343, -0.09509563, 0.050792497, -0.044465393, -0.011374812, -0.007059593, 0.088741235, -0.027865024, -0.04951675, -0.0034315472, 0.08393785, 0.058501676, -0.028675195, 0.0247696, 0.09143204, 0.0053728754, -0.09389513, 0.0005517581, 0.09601354, -0.0014464714, 0.03943989, 0.037695456, 0.030864185, -0.013616156, -0.026002366, -0.006122748, ...]","[10.004009, 3.5371523, 6.0271144, 3.9280827, 4.0390334, 5.258532, 6.2086325, 3.3145843, 5.5506644, 5.1577153, 5.1971, 5.0803313, 5.5511637, 5.394751, 5.6581945, 3.7449138, 2.4034836, 5.1774797, 5.0892663, 2.990074, 4.9095197, 4.5919943, 5.1398377, 3.8922205, 5.286442, 4.6576896, 2.5970135, 4.56016, 2.896442, 6.954224, 7.3798375, 1.98

to check no. of clusters per window

In [26]:
print("CLUSTERS PER WINDOW:")#temp

clustered_sentences_df.groupby("window_start")["cluster"].nunique().describe()
clustered_sentences_df.groupby("window_start")["cluster"].nunique().head(10)


CLUSTERS PER WINDOW:


,cluster
window_start,
2023-10-04 14:16:02,5
2023-10-05 14:16:02,3
2023-10-06 14:16:02,3
2023-10-07 14:16:02,4
2023-10-08 14:16:02,3
2023-10-09 14:16:02,3
2023-10-10 14:16:02,4
2023-10-11 14:16:02,18
2023-10-12 14:16:02,15


# **phase 4.2 :narrative continuity across windows**

compute centroids for each window-cluster

What this does (conceptually):

Takes each cluster in each window,
Computes its “average meaning”,
Stores it with window + size

In [27]:
from sklearn.metrics.pairwise import cosine_similarity

cluster_centroids = []

for (window_start, cluster_id), group in clustered_sentences_df.groupby(
    ["window_start", "cluster"]
):
    if cluster_id == -1:
        continue  # ignore noise

    embeddings = np.vstack(group["embedding_reduced"].values)
    centroid = embeddings.mean(axis=0)

    cluster_centroids.append({
        "window_start": window_start,
        "cluster": cluster_id,
        "centroid": centroid,
        "size": len(group)
    })

centroids_df = pd.DataFrame(cluster_centroids)


In [28]:
print(centroids_df.shape)
print(centroids_df.groupby("window_start").size().describe())


(1099, 4)
count    194.000000
mean       5.664948
std        4.293157
min        2.000000
25%        2.000000
50%        4.000000
75%        8.000000
max       21.000000
dtype: float64


link clusters across adjacent windows

What this does :
First window: everything is “new”
Each next window:
Compare clusters to previous window
If meaning is similar → same narrative
If not → new narrative

In [29]:
from sklearn.metrics.pairwise import cosine_similarity

BASE_SIM_THRESHOLD = 0.70        # stricter base match
STRONG_SIM_THRESHOLD = 0.75      # very strict for long narratives
MIN_CLUSTER_SIZE = 10
MAX_LOOKBACK_WINDOWS = 2        # only match to very recent windows

narrative_id = 0
narrative_map = {}

centroids_df = centroids_df.sort_values("window_start")

# Store last seen window per narrative
narrative_last_seen = {}

previous_centroids_by_window = {}

for window_start, window_group in centroids_df.groupby("window_start"):

    # Keep only recent windows for matching
    recent_windows = sorted(previous_centroids_by_window.keys())[-MAX_LOOKBACK_WINDOWS:]

    if len(recent_windows) == 0:
        # First window: start fresh narratives
        for _, row in window_group.iterrows():
            narrative_map[(window_start, row["cluster"])] = narrative_id
            narrative_last_seen[narrative_id] = window_start
            narrative_id += 1
    else:
        # Build candidate pool from recent windows
        candidate_rows = []
        candidate_keys = []

        for w in recent_windows:
            for _, r in previous_centroids_by_window[w].iterrows():
                candidate_rows.append(r)
                candidate_keys.append((r["window_start"], r["cluster"]))

        candidate_centroids = np.vstack([r["centroid"] for r in candidate_rows])

        for _, row in window_group.iterrows():
            current_centroid = row["centroid"]

            sims = cosine_similarity([current_centroid], candidate_centroids)[0]
            max_sim = sims.max()
            matched_idx = sims.argmax()
            matched_row = candidate_rows[matched_idx]
            matched_key = candidate_keys[matched_idx]

            matched_narrative = narrative_map[matched_key]

            # How long has this narrative existed?
            age = (window_start - narrative_last_seen[matched_narrative]).days

            # Choose threshold based on age
            if age > 10:
                threshold = STRONG_SIM_THRESHOLD
            else:
                threshold = BASE_SIM_THRESHOLD

            if (
                max_sim >= threshold and
                row["size"] >= MIN_CLUSTER_SIZE and
                matched_row["size"] >= MIN_CLUSTER_SIZE
            ):
                narrative_map[(window_start, row["cluster"])] = matched_narrative
                narrative_last_seen[matched_narrative] = window_start
            else:
                narrative_map[(window_start, row["cluster"])] = narrative_id
                narrative_last_seen[narrative_id] = window_start
                narrative_id += 1

    previous_centroids_by_window[window_start] = window_group


# Assign narrative IDs back

def get_narrative_id(row):
    return narrative_map.get((row["window_start"], row["cluster"]))

clustered_sentences_df["narrative_id"] = clustered_sentences_df.apply(
    get_narrative_id, axis=1
)

print("Narratives created:", clustered_sentences_df["narrative_id"].nunique())


Narratives created: 226


to see all no. of all narratives(raw ones)


In [30]:
print(clustered_sentences_df["narrative_id"].nunique())
print(clustered_sentences_df["narrative_id"].isna().sum())


226
0


attach persistent narrative IDs back to sentences

In [31]:
# 🔴 Assign narrative_id back to each sentence

def get_narrative_id(row):
    return narrative_map.get((row["window_start"], row["cluster"]))

clustered_sentences_df["narrative_id"] = clustered_sentences_df.apply(
    get_narrative_id, axis=1
)


In [32]:
clustered_sentences_df.groupby("window_start")["cluster"].nunique().describe()


,cluster
count,194.000000
mean,5.664948
std,4.293157
min,2.000000
25%,2.000000
50%,4.000000
75%,8.000000
max,21.000000


# **phase 5:narrative justification & evidence layer**

Aggregate Narrative Statistics

In [33]:
narrative_stats = (
    clustered_sentences_df
    .drop_duplicates(subset=["sentence", "timestamp", "post_id"])
    .groupby("narrative_id")
    .agg(
        total_sentences=("sentence", "count"),
        unique_posts=("post_id", "nunique"),
        windows_present=("window_start", "nunique"),
        first_seen=("timestamp", "min"),
        last_seen=("timestamp", "max")
    )
    .reset_index()
)

print("TOTAL RAW NARRATIVES:", narrative_stats.shape[0])

narrative_stats.sort_values(
    ["windows_present", "total_sentences"],
    ascending=False
).head(10)


TOTAL RAW NARRATIVES: 199


,narrative_id,total_sentences,unique_posts,windows_present,first_seen,last_seen
96,108,6076,1016,97,2024-01-06 14:51:56,2024-04-21 23:41:42
3,3,10004,1304,87,2023-10-06 20:17:48,2024-01-01 14:14:47
133,148,106,42,19,2024-02-20 18:24:28,2024-03-15 03:37:32
86,98,484,89,17,2023-12-24 06:07:48,2024-01-11 12:01:19
160,181,102,38,12,2024-03-25 16:26:28,2024-04-07 16:08:21
82,93,84,39,10,2023-12-18 16:23:17,2023-12-29 19:11:10
177,199,197,61,8,2024-04-13 15:46:10,2024-04-21 13:22:36
85,97,82,35,8,2023-12-23 16:08:30,2023-12-31 09:02:24
112,126,67,25,7,2024-02-02 20:38:24,2024-02-15 03:25:27
1,1,63,37,6,2023-10-07 08:13:25,2023-10-14 07:51:53


Qualification Rules (Transparent Filtering)

In [34]:
qualified_narratives = narrative_stats[
    (narrative_stats["total_sentences"] >= 40) &
    (narrative_stats["windows_present"] >= 3) &
    (narrative_stats["unique_posts"] >= 3)
]

print("QUALIFIED (DOMINANT) NARRATIVES:", qualified_narratives.shape[0])

qualified_narratives.sort_values(
    ["windows_present", "total_sentences"],
    ascending=False
).head(10)


QUALIFIED (DOMINANT) NARRATIVES: 12


,narrative_id,total_sentences,unique_posts,windows_present,first_seen,last_seen
96,108,6076,1016,97,2024-01-06 14:51:56,2024-04-21 23:41:42
3,3,10004,1304,87,2023-10-06 20:17:48,2024-01-01 14:14:47
133,148,106,42,19,2024-02-20 18:24:28,2024-03-15 03:37:32
86,98,484,89,17,2023-12-24 06:07:48,2024-01-11 12:01:19
160,181,102,38,12,2024-03-25 16:26:28,2024-04-07 16:08:21
82,93,84,39,10,2023-12-18 16:23:17,2023-12-29 19:11:10
177,199,197,61,8,2024-04-13 15:46:10,2024-04-21 13:22:36
85,97,82,35,8,2023-12-23 16:08:30,2023-12-31 09:02:24
112,126,67,25,7,2024-02-02 20:38:24,2024-02-15 03:25:27
1,1,63,37,6,2023-10-07 08:13:25,2023-10-14 07:51:53


STANCE SCORE TO SENTENCES(THIS IS JUST FOR THE POC , WE NEED TO IMPROVE THIS FOR THE FINAL PRODUCT

In [35]:
# ==========================================
# CELL 74 — STANCE SCORING LAYER
# ==========================================

from nltk.sentiment import SentimentIntensityAnalyzer
import nltk

nltk.download('vader_lexicon')

sia = SentimentIntensityAnalyzer()

def stance_score(s):
    # Rough stance / polarity proxy
    return sia.polarity_scores(s)["compound"]

# Add stance score to ALL clustered sentences
clustered_sentences_df["stance"] = clustered_sentences_df["sentence"].apply(stance_score)

print("Stance column added.")
print(clustered_sentences_df[["sentence", "stance"]].sample(5))


[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


Stance column added.
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     sentence  \
22843                                                                                                                                                                                                                                                              I truly believed that Palestine was empty and not valuable land like neighbouring oil rich counties, that Palestinians refused to accept reasonable peace agreements and that a

PHASE 5.5 — FRAME SPLITTING
This will:
(FRAME INDUCTION CORE FUNCTION)
1.   take top big narratives AND split them into frames
2.   assign frame_id



In [36]:
# ==========================================
# CELL 75 — FRAME INDUCTION CORE FUNCTION
# ==========================================

import hdbscan
from sklearn.preprocessing import StandardScaler

def induce_frames_for_narrative(narrative_df,
                                min_frame_size=10,
                                stance_weight=0.2):

    # Semantic embeddings
    X_embed = np.vstack(narrative_df["embedding"].values)

    # Stance dimension (scaled)
    stance = narrative_df["stance"].values.reshape(-1, 1) * stance_weight

    # Time dimension (normalized inside narrative)
    t0 = narrative_df["timestamp"].min()
    time_norm = (
        (narrative_df["timestamp"] - t0)
        .dt.total_seconds()
        / (24 * 3600)
    ).values.reshape(-1, 1)

    # Final feature vector: [semantic + stance + time]
    X = np.hstack([X_embed, stance, time_norm])

    # Normalize all dimensions
    X = StandardScaler().fit_transform(X)

    # Density-based clustering (automatic number of frames)
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=min_frame_size,
        min_samples=3,
        metric="euclidean"
    )

    labels = clusterer.fit_predict(X)

    narrative_df = narrative_df.copy()
    narrative_df["frame_id"] = labels

    # Remove noise frames
    narrative_df = narrative_df[narrative_df["frame_id"] != -1]

    return narrative_df


APPLY FRAME INDUCTION TO ALL QUALIFIED NARRATIVES

In [37]:
# ==========================================
# CELL 76 — GLOBAL FRAME INDUCTION (FINAL, FIXED INDEXING)
# ==========================================

ALL_FRAMES = []
GLOBAL_FRAME_COUNTER = 0

BIG_NARRATIVES = qualified_narratives["narrative_id"].tolist()

print("Qualified narratives to split:", BIG_NARRATIVES)

for nid in BIG_NARRATIVES:
    nd = clustered_sentences_df[
        clustered_sentences_df["narrative_id"] == nid
    ]

    if len(nd) < 80:
        print(f"Skipping narrative {nid} (too small)")
        continue

    framed = induce_frames_for_narrative(
        nd,
        min_frame_size=10,
        stance_weight=0.2
    )

    # Assign GLOBAL frame ids (unique across all narratives)
    framed = framed.copy()
    framed["global_frame_id"] = -1

    local_frames = sorted(framed["frame_id"].unique())

    frame_map = {}
    for fid in local_frames:
        frame_map[fid] = GLOBAL_FRAME_COUNTER
        GLOBAL_FRAME_COUNTER += 1

    framed["global_frame_id"] = framed["frame_id"].map(frame_map)

    print(
        f"Narrative {nid}: "
        f"{len(local_frames)} frames, "
        f"{len(framed)} sentences"
    )

    ALL_FRAMES.append(framed)

frames_df = pd.concat(ALL_FRAMES, ignore_index=True)

print("\nTOTAL FRAMES CREATED (GLOBAL):", frames_df["global_frame_id"].nunique())
print("Total framed sentences:", len(frames_df))


Qualified narratives to split: [1, 3, 93, 97, 98, 108, 124, 126, 148, 181, 199, 219]
Narrative 1: 2 frames, 99 sentences
Narrative 3: 3 frames, 25724 sentences
Narrative 93: 2 frames, 48 sentences
Narrative 97: 3 frames, 129 sentences
Narrative 98: 2 frames, 1196 sentences
Narrative 108: 3 frames, 16055 sentences
Narrative 124: 2 frames, 44 sentences
Narrative 126: 4 frames, 74 sentences
Narrative 148: 3 frames, 184 sentences
Narrative 181: 2 frames, 202 sentences
Narrative 199: 2 frames, 448 sentences
Narrative 219: 2 frames, 86 sentences

TOTAL FRAMES CREATED (GLOBAL): 30
Total framed sentences: 44289


In [38]:
# ==========================================
# CELL 78 — DEDUPLICATE SENTENCES PER FRAME
# ==========================================

frames_df = frames_df.drop_duplicates(
    subset=["narrative_id", "global_frame_id", "sentence"]
).reset_index(drop=True)

print("After deduplication, total framed sentences:", len(frames_df))


After deduplication, total framed sentences: 17402


In [39]:
# ==========================================
# CELL 77 — FRAME INSPECTION TOOL (FINAL)
# ==========================================

def inspect_frames(nid, n_samples=10):
    frames = frames_df[frames_df["narrative_id"] == nid]

    print("Narrative", nid)
    print("Frames found:", frames["frame_id"].nunique())

    for fid in sorted(frames["frame_id"].unique()):
        sub = frames[frames["frame_id"] == fid]
        gfid = sub["global_frame_id"].iloc[0]

        print(f"\n=== FRAME {fid} (GLOBAL {gfid}) ===\n")

        sample = sub.sample(
            min(n_samples, len(sub)),
            random_state=1
        )

        for s in sample["sentence"].tolist():
            print("-", s)


In [40]:
inspect_frames(3)



Narrative 3
Frames found: 3

=== FRAME 0 (GLOBAL 2) ===

- There were reconnaissance photos, elaborate maps and charts, and even taped phone conversations between senior members of Iraq's military.
- Like any map during a military operation it’s a good place to see where troops were in the past.
- I’m sure the military have satellites watching that whole area.
- Easy enough for a satellite to track the bomb that hit the hospital.

=== FRAME 1 (GLOBAL 3) ===

- Palestinians had 17 years to turn Gaza into a BEACH FRONT OASIS rivaling Dubai.
- The non-Hamas people of Gaza have nowhere to go.
- While many Palestinians are against a 2-state solution, especially those in Gaza, I can’t fault them for that.
- Its pretty easy to sit there and watch a 2 minute video of the absolutw carnage happening gaza.
- Fuck Israel and fuck everyone who makes excuses for them.
- &nbsp;  
These rallies and online comments read as support for Hamas.
- If that was truly their aim they could kill a million Gazan

In [41]:
# ==========================================
# CELL 79 — FRAME SUMMARY TABLE (FOR DASHBOARD)
# ==========================================

frame_stats = (
    frames_df
    .groupby(["narrative_id", "global_frame_id"])
    .agg(
        total_sentences=("sentence", "count"),
        unique_posts=("post_id", "nunique"),
        windows_present=("window_start", "nunique"),
        first_seen=("timestamp", "min"),
        last_seen=("timestamp", "max")
    )
    .reset_index()
)

frame_stats = frame_stats.sort_values(
    ["windows_present", "total_sentences"],
    ascending=False
)

frame_stats.head(10)


,narrative_id,global_frame_id,total_sentences,unique_posts,windows_present,first_seen,last_seen
13,108,13,6348,1033,97,2024-01-06 14:51:56,2024-04-21 23:41:42
3,3,3,9956,1297,87,2023-10-07 05:15:35,2024-01-01 14:14:47
10,98,10,516,101,17,2023-12-24 02:11:27,2024-01-11 12:01:19
22,148,22,30,15,13,2024-02-21 16:43:17,2024-03-12 23:57:05
24,181,24,68,30,12,2024-03-25 16:26:28,2024-04-07 16:08:21
21,148,21,26,15,11,2024-02-19 16:07:24,2024-03-15 03:37:32
26,199,26,187,60,8,2024-04-11 18:46:12,2024-04-21 13:22:36
8,97,8,36,21,8,2023-12-21 22:43:45,2023-12-31 09:02:24
6,93,6,14,12,7,2023-12-17 20:24:38,2023-12-29 19:11:10
0,1,0,37,27,6,2023-10-07 08:13:25,2023-10-14 07:51:53


In [42]:
inspect_frames(108)


Narrative 108
Frames found: 3

=== FRAME 0 (GLOBAL 12) ===

- All else aside if this is successful it will be a pretty gnarly feat for the US military.
- You want to be worried about a new technology and it's military/police applications, here's one for you : *flying drones* !
- Like, if there exists *any* military potential in a given technology, I would honestly expect some government to try it.
- **Military applications :** okay instead of a person firing a gun we have a robot firing a gun.

=== FRAME 1 (GLOBAL 13) ===

- Millions of dollars a day to Israel, $1 meals to be divided amongst 10,000 Palestinians.
- Witnesses said the accident happened on Friday morning near the coastal refugee camp known as al-Shati, one of the most devastated parts of Gaza, after a parachute attached to the pallet failed to deploy properly and the parcel fell on a group of men, teenagers and younger children hoping to obtain food and other supplies.
- It is while to me that people defend Hamas, and tha

Select Representative Evidence

1.   Pick sentences closest to the narrative centroid
2.   These are the most “typical” expressions



In [43]:
evidence_records = []

for _, row in qualified_narratives.iterrows():
    nid = row["narrative_id"]

    narrative_sentences = clustered_sentences_df[
        clustered_sentences_df["narrative_id"] == nid
    ]

    centroid = np.vstack(
        narrative_sentences["embedding_reduced"].values
    ).mean(axis=0)

    sims = cosine_similarity(
        np.vstack(narrative_sentences["embedding_reduced"].values),
        [centroid]
    ).flatten()

    narrative_sentences = narrative_sentences.copy()
    narrative_sentences["similarity"] = sims

    top_evidence = (
    narrative_sentences
    .drop_duplicates(subset=["sentence", "post_id", "timestamp"])
    .sort_values("similarity", ascending=False)
    .head(10)
)


    for _, ev in top_evidence.iterrows():
        evidence_records.append({
            "narrative_id": nid,
            "sentence": ev["sentence"],
            "timestamp": ev["timestamp"],
            "post_id": ev["post_id"],
            "source": ev["source"]
        })

evidence_df = pd.DataFrame(evidence_records)


Model Auditing(part of phase 5)

Build a Clean Narrative Summary Table.

which will give us:

1.   How many narratives exist?

2.   Which ones dominate?

Which ones persist?













In [44]:
narrative_summaries = qualified_narratives.copy()
narrative_summaries["duration_days"] = (
    narrative_summaries["last_seen"] - narrative_summaries["first_seen"]
).dt.days

narrative_summaries = narrative_summaries.sort_values(
    "total_sentences", ascending=False
)

narrative_summaries[
    [
        "narrative_id",
        "first_seen",
        "last_seen",
        "duration_days",
        "total_sentences",
        "unique_posts",
        "windows_present"
    ]
]


,narrative_id,first_seen,last_seen,duration_days,total_sentences,unique_posts,windows_present
3,3,2023-10-06 20:17:48,2024-01-01 14:14:47,86,10004,1304,87
96,108,2024-01-06 14:51:56,2024-04-21 23:41:42,106,6076,1016,97
86,98,2023-12-24 06:07:48,2024-01-11 12:01:19,18,484,89,17
177,199,2024-04-13 15:46:10,2024-04-21 13:22:36,7,197,61,8
133,148,2024-02-20 18:24:28,2024-03-15 03:37:32,23,106,42,19
160,181,2024-03-25 16:26:28,2024-04-07 16:08:21,12,102,38,12
82,93,2023-12-18 16:23:17,2023-12-29 19:11:10,11,84,39,10
85,97,2023-12-23 16:08:30,2023-12-31 09:02:24,7,82,35,8
112,126,2024-02-02 20:38:24,2024-02-15 03:25:27,12,67,25,7
194,219,2024-04-19 02:54:16,2024-04-22 10:14:09,3,66,26,4


Inspect One Narrative Manually (Critical Sanity Check)

 inspection cell to see what will be displayed on dashboard (engine pov)

In [45]:
# ==========================================
# ENGINE VIEW 1 — NARRATIVE LIST (WHAT DASHBOARD GETS)
# ==========================================
# Each row = ONE narrative storyline.

engine_narratives = qualified_narratives[[
    "narrative_id",
    "first_seen",
    "last_seen",
    "windows_present",
    "unique_posts",
    "total_sentences"
]].sort_values(
    ["windows_present", "total_sentences"],
    ascending=False
)

engine_narratives.head(10)


,narrative_id,first_seen,last_seen,windows_present,unique_posts,total_sentences
96,108,2024-01-06 14:51:56,2024-04-21 23:41:42,97,1016,6076
3,3,2023-10-06 20:17:48,2024-01-01 14:14:47,87,1304,10004
133,148,2024-02-20 18:24:28,2024-03-15 03:37:32,19,42,106
86,98,2023-12-24 06:07:48,2024-01-11 12:01:19,17,89,484
160,181,2024-03-25 16:26:28,2024-04-07 16:08:21,12,38,102
82,93,2023-12-18 16:23:17,2023-12-29 19:11:10,10,39,84
177,199,2024-04-13 15:46:10,2024-04-21 13:22:36,8,61,197
85,97,2023-12-23 16:08:30,2023-12-31 09:02:24,8,35,82
112,126,2024-02-02 20:38:24,2024-02-15 03:25:27,7,25,67
1,1,2023-10-07 08:13:25,2023-10-14 07:51:53,6,37,63


In [46]:
# ==========================================
# ENGINE VIEW 2 — FRAMES INSIDE ONE NARRATIVE
# ==========================================
# Each row = ONE frame inside that narrative.

NID = 108   # try narrative 3

engine_frames = frame_stats[
    frame_stats["narrative_id"] == NID
][[
    "narrative_id",
    "global_frame_id",
    "first_seen",
    "last_seen",
    "windows_present",
    "unique_posts",
    "total_sentences"
]].sort_values(
    ["windows_present", "total_sentences"],
    ascending=False
)

engine_frames


,narrative_id,global_frame_id,first_seen,last_seen,windows_present,unique_posts,total_sentences
13,108,13,2024-01-06 14:51:56,2024-04-21 23:41:42,97,1033,6348
14,108,14,2024-01-13 21:39:55,2024-02-23 11:42:20,3,3,5
12,108,12,2024-03-05 17:36:58,2024-03-09 19:38:18,2,2,4


In [47]:
# ==========================================
# ENGINE VIEW 3 — EVIDENCE FOR ONE FRAME
# ==========================================
# Each row = ONE PIECE OF EVIDENCE the journalist sees.

NID = 3
FID = 4   # try one of your real global_frame_id values

engine_evidence = frames_df[
    (frames_df["narrative_id"] == NID) &
    (frames_df["global_frame_id"] == FID)
][[
    "sentence",
    "timestamp",
    "post_id",
    "source",
    "window_start",
    "stance"
]].sort_values("timestamp")

engine_evidence.head(20)


,sentence,timestamp,post_id,source,window_start,stance
1622,"No, they probably don’t need to carpet bomb the strip to achieve it.",2023-10-12 08:03:07,175pqjh,comment,2023-10-09 14:16:02,-0.6597
6350,"Back in ww2, the only way was carpet bombing",2023-11-09 21:42:22,17rhaqa,comment,2023-11-07 14:16:02,0.0000
7830,Unrelenting carpet bombing of one of the most densely populated places on the planet where over 50% of the population are under 18.,2023-11-26 00:06:50,183vre6,comment,2023-11-23 14:16:02,0.0000
9701,"Let's be real here, is a carpet bomb necessary?",2023-12-20 14:29:53,18mrw8s,comment,2023-12-18 14:16:02,-0.4939


EXPORT LAYER FOR DASHBOARD


In [48]:
# ==========================================
# EXPORT 1 — ENGINE NARRATIVE TABLE
# ==========================================

engine_narratives = qualified_narratives[[
    "narrative_id",
    "first_seen",
    "last_seen",
    "windows_present",
    "unique_posts",
    "total_sentences"
]].sort_values(
    ["windows_present", "total_sentences"],
    ascending=False
).reset_index(drop=True)

print("ENGINE NARRATIVES PREVIEW:")
engine_narratives.head(10)


ENGINE NARRATIVES PREVIEW:


,narrative_id,first_seen,last_seen,windows_present,unique_posts,total_sentences
0,108,2024-01-06 14:51:56,2024-04-21 23:41:42,97,1016,6076
1,3,2023-10-06 20:17:48,2024-01-01 14:14:47,87,1304,10004
2,148,2024-02-20 18:24:28,2024-03-15 03:37:32,19,42,106
3,98,2023-12-24 06:07:48,2024-01-11 12:01:19,17,89,484
4,181,2024-03-25 16:26:28,2024-04-07 16:08:21,12,38,102
5,93,2023-12-18 16:23:17,2023-12-29 19:11:10,10,39,84
6,199,2024-04-13 15:46:10,2024-04-21 13:22:36,8,61,197
7,97,2023-12-23 16:08:30,2023-12-31 09:02:24,8,35,82
8,126,2024-02-02 20:38:24,2024-02-15 03:25:27,7,25,67
9,1,2023-10-07 08:13:25,2023-10-14 07:51:53,6,37,63


In [49]:
# ==========================================
# EXPORT 2 — ENGINE FRAME TABLE
# ==========================================

engine_frames = frame_stats[[
    "narrative_id",
    "global_frame_id",
    "first_seen",
    "last_seen",
    "windows_present",
    "unique_posts",
    "total_sentences"
]].sort_values(
    ["windows_present", "total_sentences"],
    ascending=False
).reset_index(drop=True)

print("ENGINE FRAMES PREVIEW:")
engine_frames.head(10)


ENGINE FRAMES PREVIEW:


,narrative_id,global_frame_id,first_seen,last_seen,windows_present,unique_posts,total_sentences
0,108,13,2024-01-06 14:51:56,2024-04-21 23:41:42,97,1033,6348
1,3,3,2023-10-07 05:15:35,2024-01-01 14:14:47,87,1297,9956
2,98,10,2023-12-24 02:11:27,2024-01-11 12:01:19,17,101,516
3,148,22,2024-02-21 16:43:17,2024-03-12 23:57:05,13,15,30
4,181,24,2024-03-25 16:26:28,2024-04-07 16:08:21,12,30,68
5,148,21,2024-02-19 16:07:24,2024-03-15 03:37:32,11,15,26
6,199,26,2024-04-11 18:46:12,2024-04-21 13:22:36,8,60,187
7,97,8,2023-12-21 22:43:45,2023-12-31 09:02:24,8,21,36
8,93,6,2023-12-17 20:24:38,2023-12-29 19:11:10,7,12,14
9,1,0,2023-10-07 08:13:25,2023-10-14 07:51:53,6,27,37


CELL 1

In [79]:
# ==========================================
# FRAME LABELING — THEME EXTRACTION SETUP (FINAL, RELIABLE)
# ==========================================

from sklearn.feature_extraction.text import TfidfVectorizer
import re
import numpy as np

# Conflict-relevant vocabulary to bias labeling
CONFLICT_TERMS = [
    "israel", "palestine", "gaza", "hamas", "idf", "ceasefire", "hostage",
    "genocide", "occupation", "bombing", "strike", "drone", "settlement",
    "netanyahu", "biden", "zionism", "antisemitism", "aid", "refugee",
    "war crimes", "civilians", "children", "hospital", "surveillance",
    "missile", "iran", "hezbollah", "protest", "sanctions"
]

print("Theme extraction setup ready ✅")


Theme extraction setup ready ✅


cell2

In [80]:
# ==========================================
# FRAME LABELING — EXTRACT KEY THEMES PER FRAME
# ==========================================

def extract_frame_keywords(sentences, top_k=6):

    # Join all sentences in frame
    text = " ".join(sentences).lower()

    # Clean
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-z\s]", " ", text)

    # TF-IDF
    vectorizer = TfidfVectorizer(
        stop_words="english",
        ngram_range=(1, 2),
        max_features=1000
    )

    X = vectorizer.fit_transform([text])
    features = np.array(vectorizer.get_feature_names_out())
    scores = X.toarray()[0]

    top_idx = scores.argsort()[::-1][:top_k]
    keywords = features[top_idx].tolist()

    return keywords


FRAME_KEYWORDS = {}

for (nid, fid), sentences in FRAME_SAMPLES.items():
    kws = extract_frame_keywords(sentences)
    FRAME_KEYWORDS[(nid, fid)] = kws

print("Extracted keywords for", len(FRAME_KEYWORDS), "frames ✅")


Extracted keywords for 30 frames ✅


CELL3


In [90]:
# ==========================================
# FRAME LABELING — FINAL SEMANTIC HEADLINE BUILDER (PRODUCTION)
# ==========================================

def build_headline_from_keywords(keywords):

    text = " ".join(keywords)

    # Normalize
    t = text.lower()

    # ---------- FRAME TYPE DETECTION (RULED SEMANTIC LAYER) ----------

    # 1. Military escalation / bombing / strikes
    if any(x in t for x in ["bomb", "bombing", "strike", "missile", "attack", "carpet", "airstrike"]):
        return "Israeli strikes and military escalation"

    # 2. Civilian casualties / hospitals / children
    if any(x in t for x in ["civilian", "children", "hospital", "dead", "killed", "massacre"]):
        return "Civilian casualties and humanitarian toll"

    # 3. Ceasefire / hostage negotiations
    if any(x in t for x in ["ceasefire", "hostage", "prisoner", "exchange", "truce"]):
        return "Ceasefire and hostage negotiations"

    # 4. US / Biden policy framing
    if "biden" in t or "united states" in t or "us support" in t:
        return "Biden policy on Israel and Gaza"

    # 5. Netanyahu / Israeli leadership
    if "netanyahu" in t or "government" in t or "cabinet" in t:
        return "Netanyahu leadership and war strategy"

    # 6. Iran / regional escalation
    if any(x in t for x in ["iran", "hezbollah", "lebanon", "syria", "proxy"]):
        return "Regional escalation involving Iran and allies"

    # 7. Occupation / resistance framing
    if any(x in t for x in ["occupation", "resistance", "freedom", "colonial", "oppression"]):
        return "Debate over occupation and armed resistance"

    # 8. Genocide / war crimes accusations
    if any(x in t for x in ["genocide", "war crime", "ethnic cleansing", "atrocity"]):
        return "Accusations of genocide and war crimes"

    # 9. Antisemitism / Zionism discourse
    if any(x in t for x in ["antisemitism", "zionism", "anti zionist", "jewish hatred"]):
        return "Antisemitism and anti-Zionism debate"

    # 10. Humanitarian crisis / aid
    if any(x in t for x in ["aid", "refugee", "food", "starvation", "humanitarian"]):
        return "Humanitarian crisis and aid delivery"

    # 11. Surveillance / intelligence / technology
    if any(x in t for x in ["satellite", "surveillance", "intelligence", "drone", "technology", "mapping"]):
        return "Military surveillance and intelligence operations"

    # 12. Protests / activism / companies
    if any(x in t for x in ["protest", "google", "employees", "boycott", "activist"]):
        return "Protests and political activism over the war"

    # ---------- FALLBACK GENERIC THEMES ----------

    if "hamas" in t and "israel" in t:
        return "Hamas attacks and Israeli retaliation"

    if "israel" in t and "gaza" in t:
        return "Israeli military operations in Gaza"

    if "palestine" in t:
        return "Debate over Palestinian state and rights"

    # Absolute fallback (rare)
    return "Debate over Israel Palestine war"


FRAME_LABELS = {}

for (nid, fid), kws in FRAME_KEYWORDS.items():
    FRAME_LABELS[(nid, fid)] = build_headline_from_keywords(kws)

print("Final semantic headlines built for", len(FRAME_LABELS), "frames ✅")


Final semantic headlines built for 30 frames ✅


CELL4

In [91]:
# ==========================================
# MERGE FRAME TITLES INTO ENGINE_FRAMES (FINAL VERSION)
# ==========================================

# Overwrite engine_frames with labeled version
engine_frames = engine_frames.copy()

def lookup_label(row):
    return FRAME_LABELS.get(
        (row["narrative_id"], row["global_frame_id"]),
        "Public debate over the Gaza war"
    )

engine_frames["frame_title"] = engine_frames.apply(
    lookup_label,
    axis=1
)

print("ENGINE FRAMES WITH TITLES PREVIEW:")
engine_frames[["narrative_id", "global_frame_id", "frame_title"]].head(10)


ENGINE FRAMES WITH TITLES PREVIEW:


,narrative_id,global_frame_id,frame_title
0,108,13,Hamas attacks and Israeli retaliation
1,3,3,Hamas attacks and Israeli retaliation
2,98,10,Israeli military operations in Gaza
3,148,22,Biden policy on Israel and Gaza
4,181,24,Biden policy on Israel and Gaza
5,148,21,Netanyahu leadership and war strategy
6,199,26,Israeli strikes and military escalation
7,97,8,Debate over Palestinian state and rights
8,93,6,Debate over Israel Palestine war
9,1,0,Israeli strikes and military escalation


In [92]:
# ==========================================
# SAVE FINAL ENGINE FRAMES FILE (WITH LABELS)
# ==========================================

engine_frames.to_csv("engine_frames.csv", index=False)

print("FINAL FILE SAVED ✅")
print("engine_frames.csv now contains frame titles")


FINAL FILE SAVED ✅
engine_frames.csv now contains frame titles


In [93]:
engine_frames.head(10)


,narrative_id,global_frame_id,first_seen,last_seen,windows_present,unique_posts,total_sentences,frame_title
0,108,13,2024-01-06 14:51:56,2024-04-21 23:41:42,97,1033,6348,Hamas attacks and Israeli retaliation
1,3,3,2023-10-07 05:15:35,2024-01-01 14:14:47,87,1297,9956,Hamas attacks and Israeli retaliation
2,98,10,2023-12-24 02:11:27,2024-01-11 12:01:19,17,101,516,Israeli military operations in Gaza
3,148,22,2024-02-21 16:43:17,2024-03-12 23:57:05,13,15,30,Biden policy on Israel and Gaza
4,181,24,2024-03-25 16:26:28,2024-04-07 16:08:21,12,30,68,Biden policy on Israel and Gaza
5,148,21,2024-02-19 16:07:24,2024-03-15 03:37:32,11,15,26,Netanyahu leadership and war strategy
6,199,26,2024-04-11 18:46:12,2024-04-21 13:22:36,8,60,187,Israeli strikes and military escalation
7,97,8,2023-12-21 22:43:45,2023-12-31 09:02:24,8,21,36,Debate over Palestinian state and rights
8,93,6,2023-12-17 20:24:38,2023-12-29 19:11:10,7,12,14,Debate over Israel Palestine war
9,1,0,2023-10-07 08:13:25,2023-10-14 07:51:53,6,27,37,Israeli strikes and military escalation


FRAME TIERING — HYBRID BALANCED

In [94]:
# ==========================================
# FRAME TIERING — HYBRID BALANCED (PRODUCTION)
# ==========================================

def assign_frame_tier(row):
    w = row["windows_present"]
    s = row["total_sentences"]
    p = row["unique_posts"]

    # 🟢 DOMINANT: long-lasting AND high volume
    if w >= 20 and s >= 1000:
        return "Dominant"

    # 🟡 EMERGING: medium persistence or medium volume
    if (8 <= w < 20) or (200 <= s < 1000):
        return "Emerging"

    # ⚪ WEAK: short-lived and low volume
    return "Weak"


# Apply tiering
engine_frames["frame_tier"] = engine_frames.apply(assign_frame_tier, axis=1)

print("FRAME TIER DISTRIBUTION:")
print(engine_frames["frame_tier"].value_counts())

print("\nSAMPLE FRAMES WITH TIERS:")
engine_frames[
    ["narrative_id", "global_frame_id", "frame_title",
     "windows_present", "total_sentences", "frame_tier"]
].head(10)


FRAME TIER DISTRIBUTION:
frame_tier
Weak        22
Emerging     6
Dominant     2
Name: count, dtype: int64

SAMPLE FRAMES WITH TIERS:


,narrative_id,global_frame_id,frame_title,windows_present,total_sentences,frame_tier
0,108,13,Hamas attacks and Israeli retaliation,97,6348,Dominant
1,3,3,Hamas attacks and Israeli retaliation,87,9956,Dominant
2,98,10,Israeli military operations in Gaza,17,516,Emerging
3,148,22,Biden policy on Israel and Gaza,13,30,Emerging
4,181,24,Biden policy on Israel and Gaza,12,68,Emerging
5,148,21,Netanyahu leadership and war strategy,11,26,Emerging
6,199,26,Israeli strikes and military escalation,8,187,Emerging
7,97,8,Debate over Palestinian state and rights,8,36,Emerging
8,93,6,Debate over Israel Palestine war,7,14,Weak
9,1,0,Israeli strikes and military escalation,6,37,Weak


In [95]:
# ==========================================
# SAVE ENGINE FRAMES WITH TIERS
# ==========================================

engine_frames.to_csv("engine_frames.csv", index=False)

print("UPDATED engine_frames.csv SAVED ✅")
print("Columns now include: frame_title + frame_tier")


UPDATED engine_frames.csv SAVED ✅
Columns now include: frame_title + frame_tier


NARRATIVE TIERING — DERIVED FROM FRAMES

In [96]:
# ==========================================
# NARRATIVE TIERING — DERIVED FROM FRAME TIERS
# ==========================================

# Count frame tiers per narrative
frame_tier_stats = (
    engine_frames
    .groupby(["narrative_id", "frame_tier"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

# Merge with narrative stats
engine_narratives = engine_narratives.merge(
    frame_tier_stats,
    on="narrative_id",
    how="left"
)

# Fill missing tier counts
for col in ["Dominant", "Emerging", "Weak"]:
    if col not in engine_narratives.columns:
        engine_narratives[col] = 0
    engine_narratives[col] = engine_narratives[col].fillna(0)


def assign_narrative_tier(row):

    # 🟢 DOMINANT narrative:
    # has at least 1 dominant frame OR very persistent overall
    if row["Dominant"] >= 1 or row["windows_present"] >= 50:
        return "Dominant"

    # 🟡 EMERGING narrative:
    # has emerging frames or medium persistence
    if row["Emerging"] >= 1 or row["windows_present"] >= 15:
        return "Emerging"

    # ⚪ WEAK narrative:
    return "Weak"


engine_narratives["narrative_tier"] = engine_narratives.apply(
    assign_narrative_tier, axis=1
)

print("NARRATIVE TIER DISTRIBUTION:")
print(engine_narratives["narrative_tier"].value_counts())

print("\nSAMPLE NARRATIVES WITH TIERS:")
engine_narratives[
    ["narrative_id", "windows_present", "total_sentences",
     "Dominant", "Emerging", "Weak", "narrative_tier"]
].head(10)


NARRATIVE TIER DISTRIBUTION:
narrative_tier
Emerging    5
Weak        5
Dominant    2
Name: count, dtype: int64

SAMPLE NARRATIVES WITH TIERS:


,narrative_id,windows_present,total_sentences,Dominant,Emerging,Weak,narrative_tier
0,108,97,6076,1,0,2,Dominant
1,3,87,10004,1,0,2,Dominant
2,148,19,106,0,2,1,Emerging
3,98,17,484,0,1,1,Emerging
4,181,12,102,0,1,1,Emerging
5,93,10,84,0,0,2,Weak
6,199,8,197,0,1,1,Emerging
7,97,8,82,0,1,2,Emerging
8,126,7,67,0,0,4,Weak
9,1,6,63,0,0,2,Weak


In [97]:
# ==========================================
# SAVE ENGINE NARRATIVES WITH TIERS
# ==========================================

engine_narratives.to_csv("engine_narratives.csv", index=False)

print("UPDATED engine_narratives.csv SAVED ✅")
print("Columns now include: narrative_tier + frame tier counts")


UPDATED engine_narratives.csv SAVED ✅
Columns now include: narrative_tier + frame tier counts


In [98]:
engine_frames[["frame_title","frame_tier","windows_present","total_sentences"]].head(8)


,frame_title,frame_tier,windows_present,total_sentences
0,Hamas attacks and Israeli retaliation,Dominant,97,6348
1,Hamas attacks and Israeli retaliation,Dominant,87,9956
2,Israeli military operations in Gaza,Emerging,17,516
3,Biden policy on Israel and Gaza,Emerging,13,30
4,Biden policy on Israel and Gaza,Emerging,12,68
5,Netanyahu leadership and war strategy,Emerging,11,26
6,Israeli strikes and military escalation,Emerging,8,187
7,Debate over Palestinian state and rights,Emerging,8,36


NARRATIVE TITLES — FROM DOMINANT FRAMES

In [99]:
# ==========================================
# NARRATIVE TITLES — FROM DOMINANT FRAMES (PRODUCTION)
# ==========================================

# For each narrative, pick its dominant frame (if exists) and use its title

# Get dominant frames only
dominant_frames = engine_frames[engine_frames["frame_tier"] == "Dominant"]

print("DOMINANT FRAMES USED FOR NARRATIVE TITLES:")
dominant_frames[["narrative_id", "frame_title"]]


DOMINANT FRAMES USED FOR NARRATIVE TITLES:


,narrative_id,frame_title
0,108,Hamas attacks and Israeli retaliation
1,3,Hamas attacks and Israeli retaliation


In [100]:
# ==========================================
# ASSIGN NARRATIVE TITLES FROM DOMINANT FRAMES
# ==========================================

# Build mapping: narrative_id → dominant frame title
dominant_title_map = (
    dominant_frames
    .drop_duplicates(subset=["narrative_id"])
    .set_index("narrative_id")["frame_title"]
    .to_dict()
)

# Assign narrative_title
def assign_narrative_title(row):

    # If narrative has a dominant frame, use its title
    if row["narrative_id"] in dominant_title_map:
        return dominant_title_map[row["narrative_id"]]

    # Otherwise fallback:
    # use most persistent emerging frame title
    candidate_frames = engine_frames[
        engine_frames["narrative_id"] == row["narrative_id"]
    ].sort_values(
        ["frame_tier", "windows_present", "total_sentences"],
        ascending=[True, False, False]
    )

    if len(candidate_frames) > 0:
        return candidate_frames.iloc[0]["frame_title"]

    # Absolute fallback (should never happen)
    return "Public discourse on Israel Palestine war"


engine_narratives["narrative_title"] = engine_narratives.apply(
    assign_narrative_title, axis=1
)

print("NARRATIVE TITLES PREVIEW:")
engine_narratives[
    ["narrative_id", "narrative_tier", "narrative_title"]
].head(10)


NARRATIVE TITLES PREVIEW:


,narrative_id,narrative_tier,narrative_title
0,108,Dominant,Hamas attacks and Israeli retaliation
1,3,Dominant,Hamas attacks and Israeli retaliation
2,148,Emerging,Biden policy on Israel and Gaza
3,98,Emerging,Israeli military operations in Gaza
4,181,Emerging,Biden policy on Israel and Gaza
5,93,Weak,Debate over Israel Palestine war
6,199,Emerging,Israeli strikes and military escalation
7,97,Emerging,Debate over Palestinian state and rights
8,126,Weak,Civilian casualties and humanitarian toll
9,1,Weak,Israeli strikes and military escalation


In [103]:
# ==========================================
# FIX DUPLICATE NARRATIVE TITLES (DISAMBIGUATION)
# ==========================================

import pandas as pd

# Find duplicate titles
title_counts = engine_narratives["narrative_title"].value_counts()

duplicate_titles = title_counts[title_counts > 1].index.tolist()

print("DUPLICATE NARRATIVE TITLES FOUND:")
print(duplicate_titles)


def add_time_disambiguator(row):
    base_title = row["narrative_title"]

    if base_title not in duplicate_titles:
        return base_title

    # Build time tag from first_seen / last_seen
    start = pd.to_datetime(row["first_seen"]).strftime("%b %Y")
    end   = pd.to_datetime(row["last_seen"]).strftime("%b %Y")

    # If long narrative, show range
    if start != end:
        return f"{base_title} — {start} to {end}"
    else:
        return f"{base_title} — {start}"


engine_narratives["narrative_title"] = engine_narratives.apply(
    add_time_disambiguator, axis=1
)

print("\nUPDATED NARRATIVE TITLES PREVIEW:")
engine_narratives[
    ["narrative_id", "narrative_tier", "narrative_title"]
].head(12)


DUPLICATE NARRATIVE TITLES FOUND:
['Hamas attacks and Israeli retaliation', 'Biden policy on Israel and Gaza', 'Debate over Palestinian state and rights', 'Israeli strikes and military escalation']

UPDATED NARRATIVE TITLES PREVIEW:


,narrative_id,narrative_tier,narrative_title
0,108,Dominant,Hamas attacks and Israeli retaliation — Jan 2024 to Apr 2024
1,3,Dominant,Hamas attacks and Israeli retaliation — Oct 2023 to Jan 2024
2,148,Emerging,Biden policy on Israel and Gaza — Feb 2024 to Mar 2024
3,98,Emerging,Israeli military operations in Gaza
4,181,Emerging,Biden policy on Israel and Gaza — Mar 2024 to Apr 2024
5,93,Weak,Debate over Israel Palestine war
6,199,Emerging,Israeli strikes and military escalation — Apr 2024
7,97,Emerging,Debate over Palestinian state and rights — Dec 2023
8,126,Weak,Civilian casualties and humanitarian toll
9,1,Weak,Israeli strikes and military escalation — Oct 2023


In [106]:
# ==========================================
# FINAL CLEANUP — ENSURE ALL NARRATIVE TITLES HAVE TIME TAG
# ==========================================

import pandas as pd

def ensure_time_in_title(row):
    title = row["narrative_title"]

    # If already has time disambiguation, keep it
    if "—" in title:
        return title

    # Otherwise append time range
    start = pd.to_datetime(row["first_seen"]).strftime("%b %Y")
    end   = pd.to_datetime(row["last_seen"]).strftime("%b %Y")

    if start != end:
        return f"{title} — {start} to {end}"
    else:
        return f"{title} — {start}"


engine_narratives["narrative_title"] = engine_narratives.apply(
    ensure_time_in_title, axis=1
)

print("FINAL NORMALIZED NARRATIVE TITLES:")
engine_narratives[
    ["narrative_id", "narrative_tier", "narrative_title"]
]


FINAL NORMALIZED NARRATIVE TITLES:


,narrative_id,narrative_tier,narrative_title
0,108,Dominant,Hamas attacks and Israeli retaliation — Jan 2024 to Apr 2024
1,3,Dominant,Hamas attacks and Israeli retaliation — Oct 2023 to Jan 2024
2,148,Emerging,Biden policy on Israel and Gaza — Feb 2024 to Mar 2024
3,98,Emerging,Israeli military operations in Gaza — Dec 2023 to Jan 2024
4,181,Emerging,Biden policy on Israel and Gaza — Mar 2024 to Apr 2024
5,93,Weak,Debate over Israel Palestine war — Dec 2023
6,199,Emerging,Israeli strikes and military escalation — Apr 2024
7,97,Emerging,Debate over Palestinian state and rights — Dec 2023
8,126,Weak,Civilian casualties and humanitarian toll — Feb 2024
9,1,Weak,Israeli strikes and military escalation — Oct 2023


In [107]:
# ==========================================
# SAVE FINAL ENGINE NARRATIVES (FINAL VERSION)
# ==========================================

engine_narratives.to_csv("engine_narratives.csv", index=False)

print("FINAL engine_narratives.csv SAVED WITH UNIQUE TITLES ✅")


FINAL engine_narratives.csv SAVED WITH UNIQUE TITLES ✅


In [105]:
engine_narratives[["narrative_id","narrative_tier","narrative_title"]]


,narrative_id,narrative_tier,narrative_title
0,108,Dominant,Hamas attacks and Israeli retaliation — Jan 2024 to Apr 2024
1,3,Dominant,Hamas attacks and Israeli retaliation — Oct 2023 to Jan 2024
2,148,Emerging,Biden policy on Israel and Gaza — Feb 2024 to Mar 2024
3,98,Emerging,Israeli military operations in Gaza
4,181,Emerging,Biden policy on Israel and Gaza — Mar 2024 to Apr 2024
5,93,Weak,Debate over Israel Palestine war
6,199,Emerging,Israeli strikes and military escalation — Apr 2024
7,97,Emerging,Debate over Palestinian state and rights — Dec 2023
8,126,Weak,Civilian casualties and humanitarian toll
9,1,Weak,Israeli strikes and military escalation — Oct 2023


...................................

In [89]:
# ==========================================
# EXPORT 3 — ENGINE EVIDENCE TABLE
# ==========================================

engine_evidence = frames_df[[
    "narrative_id",
    "global_frame_id",
    "sentence",
    "timestamp",
    "post_id",
    "source",
    "window_start",
    "stance"
]].sort_values("timestamp").reset_index(drop=True)

print("ENGINE EVIDENCE PREVIEW:")
engine_evidence.head(10)


ENGINE EVIDENCE PREVIEW:


,narrative_id,global_frame_id,sentence,timestamp,post_id,source,window_start,stance
0,3,3,"While Israeli conflicts with Palestinians are cause of controversy and no one is in the right, firing randomly into cities is just as bad as Russia firing cruise missiles into markets.",2023-10-07 05:15:35,171wuh1,comment,2023-10-04 14:16:02,-0.9022
1,3,3,Hamas is using paragliders to infiltrate Israel... this is wild af.,2023-10-07 05:35:35,171wuh1,comment,2023-10-05 14:16:02,0.0000
2,3,3,Hamas ~~militants~~ TERRORISTS are just driving around israel and killing everyone they see... alot of civilian casualties.,2023-10-07 06:18:29,171wuh1,comment,2023-10-05 14:16:02,-0.8816
3,3,3,I don't know Hamas still got this much resources.,2023-10-07 06:31:26,171wuh1,comment,2023-10-05 14:16:02,0.0000
4,3,3,"Civilian casualties are already reported and will probably continue, due to both Hamas terror tactics and IDF airstrikes.",2023-10-07 07:46:57,171zkxz,comment,2023-10-05 14:16:02,-0.5267
5,3,3,Hamas has brought immense hardship on Gaza in the immediate future good lord,2023-10-07 08:07:46,171wuh1,comment,2023-10-05 14:16:02,0.1531
6,1,0,Fellas i don't think the US will be able to pull Israel off them this time...,2023-10-07 08:13:25,171zkxz,comment,2023-10-04 14:16:02,0.0000
7,3,3,Fellas i don't think the US will be able to pull Israel off them this time...,2023-10-07 08:13:25,171zkxz,comment,2023-10-05 14:16:02,0.0000
8,3,3,and there goes any western support for Palestine,2023-10-07 08:14:19,171yy0z,comment,2023-10-05 14:16:02,0.4019
9,3,3,I was not expecting the 5th Arab Israeli war tonight,2023-10-07 08:15:00,171zkxz,comment,2023-10-05 14:16:02,-0.5994


In [51]:
# ==========================================
# EXPORT 4 — SAVE ENGINE OUTPUTS TO CSV
# ==========================================

engine_narratives.to_csv("engine_narratives.csv", index=False)
engine_frames.to_csv("engine_frames.csv", index=False)
engine_evidence.to_csv("engine_evidence.csv", index=False)

print("EXPORT COMPLETE ✅")
print("Files created:")
print(" - engine_narratives.csv")
print(" - engine_frames.csv")
print(" - engine_evidence.csv")


EXPORT COMPLETE ✅
Files created:
 - engine_narratives.csv
 - engine_frames.csv
 - engine_evidence.csv


In [52]:
# ==========================================
# EXPORT 5 — SANITY CHECKS (CRITICAL)
# ==========================================
# This ensures your outputs are correct and safe to integrate.

print("=== SANITY CHECKS ===")

print("\nNarratives:", engine_narratives.shape)
print("Frames:", engine_frames.shape)
print("Evidence:", engine_evidence.shape)

print("\nUnique narratives in frames:",
      engine_frames["narrative_id"].nunique())

print("Unique frames total:",
      engine_frames["global_frame_id"].nunique())

print("Frames per narrative (top 10):")
print(
    engine_frames.groupby("narrative_id")["global_frame_id"]
    .nunique()
    .sort_values(ascending=False)
    .head(10)
)

print("\nEvidence rows per frame (sample):")
print(
    engine_evidence.groupby(["narrative_id", "global_frame_id"])
    .size()
    .sort_values(ascending=False)
    .head(10)
)


=== SANITY CHECKS ===

Narratives: (12, 6)
Frames: (30, 7)
Evidence: (17402, 8)

Unique narratives in frames: 12
Unique frames total: 30
Frames per narrative (top 10):
narrative_id
126    4
3      3
108    3
97     3
148    3
1      2
93     2
98     2
124    2
181    2
Name: global_frame_id, dtype: int64

Evidence rows per frame (sample):
narrative_id  global_frame_id
3             3                  9956
108           13                 6348
98            10                  516
199           26                  187
181           24                   68
219           28                   52
1             0                    37
97            8                    36
148           22                   30
              21                   26
dtype: int64


In [53]:
engine_narratives.head(5)

,narrative_id,first_seen,last_seen,windows_present,unique_posts,total_sentences
0,108,2024-01-06 14:51:56,2024-04-21 23:41:42,97,1016,6076
1,3,2023-10-06 20:17:48,2024-01-01 14:14:47,87,1304,10004
2,148,2024-02-20 18:24:28,2024-03-15 03:37:32,19,42,106
3,98,2023-12-24 06:07:48,2024-01-11 12:01:19,17,89,484
4,181,2024-03-25 16:26:28,2024-04-07 16:08:21,12,38,102


In [54]:
engine_frames.head(5)

,narrative_id,global_frame_id,first_seen,last_seen,windows_present,unique_posts,total_sentences
0,108,13,2024-01-06 14:51:56,2024-04-21 23:41:42,97,1033,6348
1,3,3,2023-10-07 05:15:35,2024-01-01 14:14:47,87,1297,9956
2,98,10,2023-12-24 02:11:27,2024-01-11 12:01:19,17,101,516
3,148,22,2024-02-21 16:43:17,2024-03-12 23:57:05,13,15,30
4,181,24,2024-03-25 16:26:28,2024-04-07 16:08:21,12,30,68
